In [ ]:
import boto3
import pandas as pd
import re
import json
import uuid
from datetime import datetime
import sagemaker
from sagemaker.session import Session
from sagemaker.clarify import SageMakerClarifyProcessor, DataConfig, ModelConfig, SHAPConfig, BiasConfig
from sagemaker.model_monitor import DataCaptureConfig, DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat
from sagemaker.processing import ProcessingInput, ProcessingOutput

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', None)

# =============================================================================
# Configuration
# =============================================================================
region = 'eu-central-1'
bucket = 'mlops-bucket-058264126563'
role = 'arn:aws:iam::058264126563:role/SageMakerExecutionRole-mlops-058264126563'
model_package_arn = 'arn:aws:sagemaker:eu-central-1:058264126563:model-package/flight-delay-model-package-group/2'

# SageMaker session
sagemaker_session = Session()
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# S3 paths

baseline_path = f"s3://{bucket}/baseline_report/"
shap_output_path = f"s3://{bucket}/shap-analysis"
monitoring_output_path = f"s3://{bucket}/monitoring-results/"
baseline_constraints_uri = f"s3://{bucket}/baseline_report/constraints.json"
baseline_statistics_uri = f"s3://{bucket}/baseline_report/statistics.json"

unique_id = str(uuid.uuid4())[:8]  # Job ismini daha benzersiz yapmak için 8 haneli UUID
monitoring_job_name = f"flight-delay-monitoring-{timestamp}-{unique_id}"
shap_job_name = f"flight-delay-shap-{timestamp}-{unique_id}"

cloudwatch = boto3.client('cloudwatch', region_name=region)
sns = boto3.client('sns', region_name=region)

sns_topic_arn = 'arn:aws:sns:eu-central-1:058264126563:MLOpsInfraStage-MLOpsNotificationStack-MLOpsNotificationTopic3FAE029C-KcXITPyFOtHW'
endpoint_name = 'mlops-prod-endpoint'

print(f"🚀 Starting Model Monitoring & SHAP Analysis - {timestamp}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
🚀 Starting Model Monitoring & SHAP Analysis - 20250910-070148


In [ ]:
# Create Endpoint for Testing Purpose

sm_client = boto3.client("sagemaker", region_name="eu-central-1")

model_name = "mlops-prod-model"
endpoint_config_name = "mlops-prod-config"
endpoint_name = "mlops-prod-endpoint"

# 1. Model 
sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "ModelPackageName": model_package_arn
    }
)

print(f"Model created: {model_name}")

# 2. Endpoint Config 
sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.c5.large",
        }
    ],
        DataCaptureConfig={
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": f"s3://{bucket}/data-capture/",
        "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
        "CaptureContentTypeHeader": {
            "CsvContentTypes": ["text/csv"],
        }
    }
)

print(f"Endpoint config created: {endpoint_config_name}")

sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name
)

print(f"Endpoint creation initiated: {endpoint_name}")

Model created: mlops-prod-model
Endpoint config created: mlops-prod-config
Endpoint creation initiated: mlops-prod-endpoint


In [ ]:
# =============================================================================
# 1. Get Feature Names from Baseline Constraints
# =============================================================================
s3_client = boto3.client('s3', region_name=region)

try:
    constraints_obj = s3_client.get_object(Bucket=bucket, Key="baseline_report/constraints.json")
    constraints_data = json.loads(constraints_obj['Body'].read().decode('utf-8'))
    feature_names = []
    target_column = 'dep_delay'
    
    for feature in constraints_data['features']:
        if feature['name'] != target_column:  
            feature_names.append(feature['name'])
    
    print(f"✅ Loaded {len(feature_names)} feature names from constraints.json (excluding target: {target_column})")
    print("Feature names:", feature_names)
    
except Exception as e:
    print(f"❌ Error reading constraints.json: {e}")
    exit(1)

✅ Loaded 28 feature names from constraints.json (excluding target: dep_delay)
Feature names: ['year', 'month', 'day', 'arr_delay', 'distance', 'hour', 'minute', 'temp', 'humid', 'wind_dir', 'wind_speed', 'precip', 'pressure', 'visib', 'distance_ratio_by_total', 'distance_category', 'aircraft_count_by_airline', 'allegiant_air', 'american_airlines_inc', 'delta_air_lines_inc', 'frontier_airlines_inc', 'hawaiian_airlines_inc', 'horizon_air', 'jetblue_airways', 'skywest_airlines_inc', 'southwest_airlines_co', 'spirit_air_lines', 'united_air_lines_inc']


In [ ]:
# =============================================================================
# 2. Load Data from Data Capture 
# =============================================================================

capture_data = []
prefix = "data-capture/mlops-prod-endpoint/AllTraffic/"

# 1️⃣ Year
resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter='/')
years = [p['Prefix'] for p in resp.get('CommonPrefixes', [])]

for year_prefix in years:
    # 2️⃣ Month
    resp_months = s3_client.list_objects_v2(Bucket=bucket, Prefix=year_prefix, Delimiter='/')
    months = [m['Prefix'] for m in resp_months.get('CommonPrefixes', [])]
    
    for month_prefix in months:
        # 3️⃣ Day
        resp_days = s3_client.list_objects_v2(Bucket=bucket, Prefix=month_prefix, Delimiter='/')
        days = [d['Prefix'] for d in resp_days.get('CommonPrefixes', [])]
        
        for day_prefix in days:
            # 4️⃣ Hour
            resp_hours = s3_client.list_objects_v2(Bucket=bucket, Prefix=day_prefix, Delimiter='/')
            hours = [h['Prefix'] for h in resp_hours.get('CommonPrefixes', [])]
            
            for hour_prefix in hours:
                # 5️⃣ JSONL files
                resp_files = s3_client.list_objects_v2(Bucket=bucket, Prefix=hour_prefix)
                if 'Contents' not in resp_files:
                    continue
                    
                jsonl_files = [obj['Key'] for obj in resp_files['Contents'] if obj['Key'].endswith('.jsonl')]
                print(f"📁 Found {len(jsonl_files)} JSONL files in {hour_prefix}")

                for key in jsonl_files:
                    try:
                        obj = s3_client.get_object(Bucket=bucket, Key=key)
                        for line in obj['Body'].read().decode('utf-8').strip().split('\n'):
                            if line:
                                capture_data.append(json.loads(line))
                    except Exception as e:
                        print(f"⚠️ Error reading {key}: {e}")

print(f"✅ Loaded {len(capture_data)} data capture records")

📁 Found 14 JSONL files in data-capture/mlops-prod-endpoint/AllTraffic/2025/09/09/15/
📁 Found 39 JSONL files in data-capture/mlops-prod-endpoint/AllTraffic/2025/09/09/16/
✅ Loaded 14189 data capture records


In [18]:
capture_data[:1]

[{'captureData': {'endpointInput': {'observedContentType': 'text/csv',
    'mode': 'INPUT',
    'data': '2022,4,1,-10.0,414.96195472435,11.0,25.0,47.99342830602247,93.96753338294226,190.0,8.119476056722446,0.0,1015.5665730904666,10.0,0.1988698877649567,0.0,4.0,0,0,0,0,0,1,0,0,0,0,0\n',
    'encoding': 'CSV'},
   'endpointOutput': {'observedContentType': 'text/csv; charset=utf-8',
    'mode': 'OUTPUT',
    'data': '-3.41\n',
    'encoding': 'CSV'}},
  'eventMetadata': {'eventId': 'ef40ea45-67e6-4aff-b230-1cd4bdb14abd',
   'inferenceTime': '2025-09-09T15:46:11Z'},
  'eventVersion': '0'}]

In [ ]:
# =============================================================================
# 3. Parse Data Capture Format and Create DataFrame
# =============================================================================

parsed_data = []

for record in capture_data:
    try:
        # Input data (CSV format) 
        input_data = record['captureData']['endpointInput']['data'].strip()
        input_values = input_data.split(',')
        
        # Output data (prediction) 
        output_data = record['captureData']['endpointOutput']['data'].strip()
        prediction = float(output_data)
        
        # Feature dictionary 
        if len(input_values) == len(feature_names):
            feature_dict = {}
            for i, name in enumerate(feature_names):
                try:
                    feature_dict[name] = float(input_values[i])
                except ValueError:
                    feature_dict[name] = input_values[i]  
            
            # 
            feature_dict[target_column] = prediction
            feature_dict['inference_time'] = record['eventMetadata']['inferenceTime']
            feature_dict['event_id'] = record['eventMetadata']['eventId']
            
            parsed_data.append(feature_dict)
        else:
            print(f"⚠️ Input length mismatch: expected {len(feature_names)}, got {len(input_values)}")
            
    except Exception as e:
        print(f"⚠️ Error parsing record: {e}")
        continue

df_captured = pd.DataFrame(parsed_data)
print(f"✅ Parsed {len(df_captured)} records from data capture")
print("Sample columns:", list(df_captured.columns))
print(f"Columns Length:", len(df_captured.columns))
df_captured.head()

✅ Parsed 14189 records from data capture
Sample columns: ['year', 'month', 'day', 'arr_delay', 'distance', 'hour', 'minute', 'temp', 'humid', 'wind_dir', 'wind_speed', 'precip', 'pressure', 'visib', 'distance_ratio_by_total', 'distance_category', 'aircraft_count_by_airline', 'allegiant_air', 'american_airlines_inc', 'delta_air_lines_inc', 'frontier_airlines_inc', 'hawaiian_airlines_inc', 'horizon_air', 'jetblue_airways', 'skywest_airlines_inc', 'southwest_airlines_co', 'spirit_air_lines', 'united_air_lines_inc', 'dep_delay', 'inference_time', 'event_id']
Columns Length: 31


,year,month,day,arr_delay,distance,hour,minute,temp,humid,wind_dir,wind_speed,precip,pressure,visib,distance_ratio_by_total,distance_category,aircraft_count_by_airline,allegiant_air,american_airlines_inc,delta_air_lines_inc,frontier_airlines_inc,hawaiian_airlines_inc,horizon_air,jetblue_airways,skywest_airlines_inc,southwest_airlines_co,spirit_air_lines,united_air_lines_inc,dep_delay,inference_time,event_id
0,2022.0,4.0,1.0,-10.0,414.961955,11.0,25.0,47.993428,93.967533,190.0,8.119476,0.00,1015.566573,10.0,0.198870,0.0,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-3.41,2025-09-09T15:46:11Z,ef40ea45-67e6-4aff-b230-1cd4bdb14abd
1,2022.0,1.0,30.0,-6.0,801.073028,20.0,0.0,46.723471,100.000000,160.0,9.537581,0.01,1010.097289,10.0,0.625079,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.84,2025-09-09T15:46:12Z,a5b77ae9-d846-48e2-a79c-df69bfb48fd3
2,2022.0,2.0,1.0,-10.0,1223.486191,10.0,15.0,43.295377,96.831519,200.0,10.358023,0.00,1025.774077,10.0,0.625079,2.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.46,2025-09-09T15:46:12Z,0bbeb2d5-924e-4887-aa96-66f94021e1e5
3,2022.0,4.0,28.0,2.0,428.738056,8.0,0.0,52.046060,100.000000,130.0,6.916666,0.00,1013.990270,10.0,0.198870,0.0,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,-1.84,2025-09-09T15:46:12Z,92fbff17-688d-41ca-a539-db1832475fc8
4,2022.0,5.0,5.0,32.0,2805.170865,15.0,30.0,49.531693,93.762807,200.0,8.500532,0.02,1005.550892,7.0,0.625079,2.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.61,2025-09-09T15:46:13Z,df43e7a8-6aae-449b-ae91-ee55f11336ec


In [ ]:
# =============================================================================
# 4. Prepare features for SHAP
# =============================================================================

# Clean data
df_clean = df_captured.dropna()
print(f"📊 Cleaned data: {len(df_clean)} records (removed {len(df_captured) - len(df_clean)} NaN rows)")

# Save processed data to S3 
processed_data_key = f"processed-data/flight-features-{timestamp}.csv"
df_for_shap = df_clean.drop(columns=['inference_time', 'event_id'])
df_for_shap.to_csv(f"s3://{bucket}/{processed_data_key}", index=False)
print(f"💾 Processed data saved to: s3://{bucket}/{processed_data_key}")

📊 Cleaned data: 14189 records (removed 0 NaN rows)
💾 Processed data saved to: s3://mlops-bucket-058264126563/processed-data/flight-features-20250910-044710.csv


In [23]:
# =============================================================================
# 5. SHAP Explainability Analysis
# =============================================================================
clarify_processor = SageMakerClarifyProcessor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    sagemaker_session=sagemaker_session
)

# Feature kolonları (label hariç)
headers = df_for_shap.columns.to_list()
feature_cols = [col for col in headers if col != target_column]

shap_data_config = DataConfig(
    s3_data_input_path=f"s3://{bucket}/{processed_data_key}",
    s3_output_path=shap_output_path,
    label=target_column,
    headers=headers,
    dataset_type="text/csv"
)

model_config = ModelConfig(
    endpoint_name="mlops-prod-endpoint",
    content_type="text/csv",
    accept_type="application/json"
)

# Baseline = mean of numeric feature columns
baseline_values = df_for_shap[feature_cols].mean().fillna(0).tolist()

shap_config = SHAPConfig(
    baseline=[baseline_values],
    num_samples=10,
    agg_method="mean_abs"
)

clarify_processor.run_explainability(
    data_config=shap_data_config,
    model_config=model_config,
    explainability_config=shap_config,
    job_name=shap_job_name,
    wait=False,
    logs=False
)

print(f"✅ SHAP job started: {shap_job_name}")
print(f"📁 Results will be saved to: {shap_output_path}")

# =============================================================================
# 6. Optional: Data Quality Report
# =============================================================================
print("\n📈 Data Quality Summary:")
print(f"Total records processed: {len(df_captured)}")
print(f"Records after cleaning: {len(df_clean)}")
print(f"Features used for SHAP: {len(feature_cols)}")
print(f"Prediction range: {df_clean[target_column].min():.2f} to {df_clean[target_column].max():.2f}")

INFO:sagemaker:Creating processing-job with name flight-delay-shap-20250910-044710-e263996e


✅ SHAP job started: flight-delay-shap-20250910-044710-e263996e
📁 Results will be saved to: s3://mlops-bucket-058264126563/shap-analysis

📈 Data Quality Summary:
Total records processed: 14189
Records after cleaning: 14189
Features used for SHAP: 28
Prediction range: -8.86 to 512.43


In [ ]:
# =============================================================================
# 7. Model Monitoring
# =============================================================================

from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator
from sagemaker.session import Session

# SageMaker session
sagemaker_session = Session()

# Monitor objesi
monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session
)

monitor_schedule_name = "flight-delay-dataquality-monitor"

monitor.create_monitoring_schedule(
    monitor_schedule_name=monitor_schedule_name,
    endpoint_input="mlops-prod-endpoint",
    statistics=baseline_statistics_uri,
    constraints=baseline_constraints_uri,
    output_s3_uri=monitoring_output_path,
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True
)

print("✅ Drift monitoring schedule created (every 1 hours)")

2025-09-09 15:45:33,584 - INFO - Ignoring unnecessary instance type: None.
2025-09-09 15:45:34,216 - INFO - Creating Monitoring Schedule with name: flight-delay-dataquality-monitor


✅ Drift monitoring schedule created (every 1 hours)


In [ ]:
import boto3

sm_client = boto3.client("sagemaker")

response = sm_client.describe_monitoring_schedule(
    MonitoringScheduleName=monitor_schedule_name
)

print("📊 Monitoring schedule status:", response["MonitoringScheduleStatus"])

📊 Monitoring schedule status: Stopped


In [ ]:
import boto3

sm_client = boto3.client("sagemaker")

monitor_schedule_name = "flight-delay-dataquality-monitor"

sm_client.stop_monitoring_schedule(
    MonitoringScheduleName=monitor_schedule_name
)

print(f"✅ Monitoring schedule '{monitor_schedule_name}' durduruldu.")

✅ Monitoring schedule 'flight-delay-dataquality-monitor' durduruldu.


In [ ]:
sm_client.start_monitoring_schedule(
    MonitoringScheduleName=monitor_schedule_name
)
print(f"✅ Monitoring schedule '{monitor_schedule_name}' tekrar başlatıldı.")

✅ Monitoring schedule 'flight-delay-dataquality-monitor' tekrar başlatıldı.


In [ ]:
sm_client.delete_monitoring_schedule(
    MonitoringScheduleName=monitor_schedule_name
)
print(f"❌ Monitoring schedule '{monitor_schedule_name}' silindi.")

❌ Monitoring schedule 'flight-delay-dataquality-monitor' silindi.


In [26]:
# =============================================================================
# 5. Compare with Existing Baseline (Drift Detection)
# =============================================================================
print("\n🚨 Checking for Data Drift...")
try:
    # Read existing baseline statistics AND constraints
    baseline_stats_obj = s3_client.get_object(Bucket=bucket, Key='baseline_report/statistics.json')
    baseline_stats = json.loads(baseline_stats_obj['Body'].read().decode('utf-8'))
    
    baseline_constraints_obj = s3_client.get_object(Bucket=bucket, Key='baseline_report/constraints.json')
    baseline_constraints = json.loads(baseline_constraints_obj['Body'].read().decode('utf-8'))
    
    print("📊 Baseline statistics loaded successfully")
    print("🔒 Baseline constraints loaded successfully")
    
    # Simple drift detection - compare means and check constraints
    print("\n🔍 Drift & Constraint Analysis:")
    
    numeric_columns = df_clean.select_dtypes(include=['float64', 'int64']).columns
    
    # Prepare report text
    report_lines = []
    report_lines.append("DRIFT & CONSTRAINT ANALYSIS REPORT")
    report_lines.append("=====================================")
    report_lines.append(f"Timestamp: {timestamp}")
    report_lines.append(f"Total Records: {len(df_clean)}")
    report_lines.append("")
    
    drift_count = 0
    violation_count = 0
    total_features = 0
    
    for col in numeric_columns:
        current_mean = float(df_clean[col].mean())
        current_min = float(df_clean[col].min())
        current_max = float(df_clean[col].max())
        
        # Find baseline stats
        baseline_mean = None
        for feature in baseline_stats.get('features', []):
            if feature.get('name') == col:
                baseline_mean = feature.get('numerical_statistics', {}).get('mean', 0)
                break
        
        # Find baseline constraints
        constraint_min = None
        constraint_max = None
        for feature in baseline_constraints.get('features', []):
            if feature.get('name') == col:
                constraints = feature.get('numerical_constraints', {})
                constraint_min = constraints.get('min_value')
                constraint_max = constraints.get('max_value')
                break
        
        if baseline_mean is not None:
            total_features += 1
            drift_pct = abs((current_mean - baseline_mean) / baseline_mean) * 100
            is_drift = drift_pct > 10
            
            # Check constraint violations
            min_violation = current_min < constraint_min if constraint_min is not None else False
            max_violation = current_max > constraint_max if constraint_max is not None else False
            has_violation = min_violation or max_violation
            
            # Determine status
            if has_violation:
                status = "🔴 VIOLATION"
                violation_count += 1
            elif is_drift:
                status = "🟥 DRIFT"
                drift_count += 1
            else:
                status = "🟢 OK"
            
            # Console output
            constraint_info = ""
            if constraint_min is not None or constraint_max is not None:
                constraint_info = f" | Limits: [{constraint_min}, {constraint_max}]"
                if min_violation:
                    constraint_info += f" | MIN_VIOL: {current_min:.2f}"
                if max_violation:
                    constraint_info += f" | MAX_VIOL: {current_max:.2f}"
            
            print(f"  {col:20}: {status} ({drift_pct:.1f}% change){constraint_info}")
            
            # Add to report
            report_lines.append(f"{col:20} | {status:12} | Drift: {drift_pct:6.1f}% | Current: {current_mean:8.4f} | Baseline: {baseline_mean:8.4f}{constraint_info}")
            
        else:
            print(f"  {col:20}: ⚪ No baseline found")
    
    # Add summary to report
    report_lines.append("")
    report_lines.append("SUMMARY:")
    report_lines.append(f"Total Features: {total_features}")
    report_lines.append(f"Constraint Violations: {violation_count}")
    report_lines.append(f"Statistical Drift: {drift_count}")
    report_lines.append(f"OK Features: {total_features - drift_count - violation_count}")
    
    # Save report to S3
    try:
        report_key = f"drift-reports/drift-constraint-analysis-{timestamp}.txt"
        report_text = "\n".join(report_lines)
        
        s3_client.put_object(
            Bucket=bucket,
            Key=report_key,
            Body=report_text,
            ContentType='text/plain'
        )
        
        print(f"\n📁 Report saved to: s3://{bucket}/{report_key}")
        print("✅ S3 upload successful!")
        
    except Exception as save_error:
        print(f"❌ Failed to save report: {str(save_error)}")
    
    # Print summary
    print(f"\n📊 SUMMARY:")
    print(f"   🔴 Constraint Violations: {violation_count}")
    print(f"   🟥 Statistical Drift: {drift_count}")
    print(f"   🟢 OK Features: {total_features - drift_count - violation_count}")
    print(f"   📋 Total Features Analyzed: {total_features}")
            
except Exception as e:
    print(f"❌ Drift analysis error: {str(e)}")


🚨 Checking for Data Drift...
📊 Baseline statistics loaded successfully
🔒 Baseline constraints loaded successfully

🔍 Drift & Constraint Analysis:
  year                : 🟢 OK (0.0% change)
  month               : 🟢 OK (0.8% change)
  day                 : 🟢 OK (0.7% change)
  arr_delay           : 🟢 OK (7.1% change)
  distance            : 🟢 OK (9.9% change)
  hour                : 🟢 OK (0.6% change)
  minute              : 🟢 OK (2.9% change)
  temp                : 🟥 DRIFT (10.9% change)
  humid               : 🟥 DRIFT (10.5% change)
  wind_dir            : 🟢 OK (0.7% change)
  wind_speed          : 🟥 DRIFT (17.7% change)
  precip              : 🟢 OK (2.1% change)
  pressure            : 🟢 OK (0.3% change)
  visib               : 🟢 OK (0.1% change)
  distance_ratio_by_total: 🟥 DRIFT (16.1% change)
  distance_category   : 🟢 OK (7.6% change)
  aircraft_count_by_airline: 🟥 DRIFT (99.2% change)
  allegiant_air       : 🟥 DRIFT (19.8% change)
  american_airlines_inc: 🟥 DRIFT (28.9% change)